# Fine-tune ViDeBERTa-xsmall — đọc gói và lưu kết quả trên Drive

Gói ZIP đã được đặt trong `My Drive/nlp_chuan_hoa_dia_chi/`. Chọn GPU rồi chạy từng mục; không dùng widget upload file. Mount Drive lần đầu có thể cần xác nhận tài khoản.

**Cấu hình mặc định:** 5 epoch, 4.841 mẫu train nhận diện và 1.169 mẫu riêng cho ranker. Không có nhãn test cuối. Checkpoint và báo cáo được lưu trên Drive. Train GPU dự trù 10–30 phút, ETA thực được in sau 10 batch.

## 1. Kiểm tra GPU

In [ ]:
import sys, subprocess, torch
assert torch.cuda.is_available(), "Bật GPU ở Runtime → Change runtime type trước khi chạy tiếp"
print("Torch:", torch.__version__, "GPU:", torch.cuda.get_device_name())
subprocess.run(["nvidia-smi"], check=True)


Torch: 2.11.0+cu130 GPU: Tesla T4


CompletedProcess(args=['nvidia-smi'], returncode=0)

## 2. Đọc gói từ Drive
Kết nối Drive rồi giải nén gói đã có vào `/content` để xử lý nhanh. Kiểm tra SHA-256 trước khi dùng. Không cần upload lại ZIP khi Colab bị ngắt.

In [8]:
from google.colab import drive
from pathlib import Path
import zipfile, os, sys, hashlib

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/nlp_chuan_hoa_dia_chi")
ZIP_PATH = DRIVE_ROOT / "nlp_dia_chi_colab_v1.zip"
assert ZIP_PATH.is_file(), f"Chưa thấy ZIP đồng bộ lên Drive: {ZIP_PATH}"
assert hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest() == "e346d0b800a315c78fbdf9c705dd2a80144dbeeae5b2427793f54f40d840bd1f", "ZIP khác bản đã kiểm tra"
PROJECT = Path("/content/nlp_dia_chi_colab")
with zipfile.ZipFile(ZIP_PATH) as z:
    for name in z.namelist():
        assert (Path("/content") / name).resolve().is_relative_to(PROJECT), name
    z.extractall("/content")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print("Đã đọc gói từ Drive:", ZIP_PATH)


Mounted at /content/drive
Đã đọc gói từ Drive: /content/drive/MyDrive/nlp_chuan_hoa_dia_chi/nlp_dia_chi_colab_v1.zip


## 3. Cài thư viện
Giữ nguyên PyTorch CUDA của Colab. Không chạy `requirements-neural.txt` vì đó là môi trường CPU cũ.

In [9]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "colab/requirements.txt"], check=True)
assert torch.cuda.is_available(), "CUDA phải còn hoạt động sau cài thư viện"


## 4. Kiểm tra gói và smoke test
Các test dùng backbone tí hon để kiểm tra gradient, lưu/nạp, Unicode và pipeline; chưa phải fine-tune đầy đủ.

In [12]:
import subprocess
import sys

# Chạy trực tiếp và xuất kết quả stderr chi tiết để chẩn đoán nguyên nhân verify_bundle thất bại
try:
    print("Đang chạy verify_bundle...")
    result = subprocess.run(
        [sys.executable, "-m", "colab.verify_bundle", "--root", str(PROJECT)],
        capture_output=True,
        text=True,
        check=True
    )
    print("STDOUT:\n", result.stdout)
    print("Verify thành công! Bắt đầu chạy test...")
    subprocess.run([sys.executable, "-m", "unittest", "tests.test_videberta_colab", "tests.test_char", "tests.test_pipeline", "tests.test_pipeline_v3", "tests.test_crf", "tests.test_danh_gia"], check=True)
except subprocess.CalledProcessError as e:
    print("--- LỖI XẢY RA KHI CHẠY VERIFY_BUNDLE ---")
    print("Mã lỗi thoát (Exit Code):", e.returncode)
    print("Đầu ra chuẩn (STDOUT):\n", e.stdout)
    print("Đầu ra lỗi (STDERR):\n", e.stderr)


Đang chạy verify_bundle...
--- LỖI XẢY RA KHI CHẠY VERIFY_BUNDLE ---
Mã lỗi thoát (Exit Code): 1
Đầu ra chuẩn (STDOUT):
 
Đầu ra lỗi (STDERR):
 Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/nlp_dia_chi_colab/colab/verify_bundle.py", line 33, in <module>
    main()
    ~~~~^^
  File "/content/nlp_dia_chi_colab/colab/verify_bundle.py", line 29, in main
    verify(args.root)
    ~~~~~~^^^^^^^^^^^
  File "/content/nlp_dia_chi_colab/colab/verify_bundle.py", line 17, in verify
    sets = validate_input(root / 'bundle_data')
  File "/content/nlp_dia_chi_colab/colab/train_videberta.py", line 49, in validate_input
    if digest(data_root / 'snapshot' / name) != h:
       ~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/nlp_dia_chi_colab/danh_gia/chay_ban_thu.py", line 42, in digest
    return hashlib.sha256(path.read_bytes()).hexdigest()
                          ~~~~~~~~~~~~~~~^

In [16]:
import re
from pathlib import Path

# 1. Sửa hàm `digest` trong file `danh_gia/chay_ban_thu.py` để tương thích Windows/Linux path
file_chay_ban_thu = Path("/content/nlp_dia_chi_colab/danh_gia/chay_ban_thu.py")
if file_chay_ban_thu.exists():
    content = file_chay_ban_thu.read_text(encoding="utf-8")
    # Thay thế hàm digest để xử lý đường dẫn chuỗi có dấu gạch chéo ngược Windows
    old_digest_func = """def digest(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()"""

    new_digest_func = """def digest(path):
    # Chuẩn hóa đường dẫn Windows backslash sang Linux forward slash
    normalized_path = Path(str(path).replace('\\\\', '/').replace('\\', '/'))
    return hashlib.sha256(normalized_path.read_bytes()).hexdigest()"""

    if "def digest(path):" in content:
        # Sử dụng biểu thức chính quy để sửa đổi linh hoạt hơn
        content_modified = re.sub(
            r"def digest\(path\):\s+return hashlib\.sha256\(path\.read_bytes\(\)\)\.hexdigest\(\)",
            new_digest_func,
            content
        )
        file_chay_ban_thu.write_text(content_modified, encoding="utf-8")
        print("Đã sửa thành công hàm digest() trong danh_gia/chay_ban_thu.py!")
    else:
        print("Không tìm thấy cấu trúc hàm digest() mặc định, hãy kiểm tra lại file.")
else:
    print("Không tìm thấy file danh_gia/chay_ban_thu.py")


Đã sửa thành công hàm digest() trong danh_gia/chay_ban_thu.py!


In [17]:
# 2. Sửa file `colab/train_videberta.py` để chuẩn hóa các đường dẫn chứa ký tự \ sang /
file_train_videberta = Path("/content/nlp_dia_chi_colab/colab/train_videberta.py")
if file_train_videberta.exists():
    content_train = file_train_videberta.read_text(encoding="utf-8")

    # Thay thế đoạn lấy tên tệp có chứa dấu gạch chéo ngược Windows
    old_block = "digest(data_root / 'snapshot' / name)"
    new_block = "digest(data_root / 'snapshot' / name.replace('\\\\', '/').replace('\\', '/'))"

    if old_block in content_train:
        content_train_modified = content_train.replace(old_block, new_block)
        file_train_videberta.write_text(content_train_modified, encoding="utf-8")
        print("Đã sửa thành công phần kiểm tra path trong colab/train_videberta.py!")
    else:
        print("Cảnh báo: Không tìm thấy dòng kiểm tra snapshot path mặc định.")


Đã sửa thành công phần kiểm tra path trong colab/train_videberta.py!


In [ ]:
import re
import json
from pathlib import Path

file_train_videberta = Path("/content/nlp_dia_chi_colab/colab/train_videberta.py")
if file_train_videberta.exists():
    content = file_train_videberta.read_text(encoding="utf-8")

    # 1. Sửa đổi triệt để hàm validate_input để chuẩn hóa đường dẫn
    old_validate = """def validate_input(data_root):
    data_root = Path(data_root)
    manifest = json.loads((data_root / 'manifest.json').read_text('utf-8'))
    sets = {}
    for subset, files in manifest.items():
        sets[subset] = {}
        for name, h in files.items():
            if digest(data_root / 'snapshot' / name) != h:
                raise ValueError(f'File {name} has incorrect digest')
            sets[subset][name] = data_root / 'snapshot' / name
    return sets"""

    new_validate = """def validate_input(data_root):
    data_root = Path(data_root)
    manifest = json.loads((data_root / 'manifest.json').read_text('utf-8'))
    sets = {}
    for subset, files in manifest.items():
        sets[subset] = {}
        for name, h in files.items():
            # Thay đổi toàn bộ dấu gạch chéo ngược Windows sang định dạng Linux
            normalized_name = name.replace('\\\\', '/').replace('\\', '/')
            target_path = data_root / 'snapshot' / normalized_name
            if digest(target_path) != h:
                raise ValueError(f'File {name} has incorrect digest')
            sets[subset][normalized_name] = target_path
    return sets"""

    if "def validate_input" in content:
        content = content.replace(old_validate, new_validate)
    else:
        # Trường hợp đã bị thay thế một phần trước đó
        content = re.sub(r"def validate_input.*?return sets", new_validate, content, flags=re.DOTALL)

    # 2. Đảm bảo các lệnh gọi truy cập tập tin snapshot sử dụng đường dẫn chuẩn hóa
    content = content.replace("sets['train']", "{k.replace('\\\\','/').replace('\\','/'): v for k, v in sets['train'].items()}")
    content = content.replace("sets['dev']", "{k.replace('\\\\','/').replace('\\','/'): v for k, v in sets['dev'].items()}")
    content = content.replace("sets['ranker_train']", "{k.replace('\\\\','/').replace('\\','/'): v for k, v in sets['ranker_train'].items()}")
    content = content.replace("sets['ranker_dev']", "{k.replace('\\\\','/').replace('\\','/'): v for k, v in sets['ranker_dev'].items()}")

    file_train_videberta.write_text(content, encoding="utf-8")
    print("Đã sửa đổi đồng bộ và chuẩn hóa hoàn tất colab/train_videberta.py!")
else:
    print("Không tìm thấy colab/train_videberta.py")


In [13]:
# Chẩn đoán chi tiết lỗi của verify_bundle
import json
from pathlib import Path

manifest_path = Path("/content/nlp_dia_chi_colab/BUNDLE_MANIFEST.json")
if manifest_path.exists():
    try:
        with open(manifest_path, "r", encoding="utf-8") as f:
            manifest = json.load(f)
        print("Đã tìm thấy BUNDLE_MANIFEST.json chứa", len(manifest.get("files", [])), "tệp cấu hình.")

        # Kiểm tra nhanh xem có tệp nào bị thiếu hoặc sai kích thước không
        missing_files = []
        for file_info in manifest.get("files", []):
            p = Path("/content/nlp_dia_chi_colab") / file_info["path"]
            if not p.exists():
                missing_files.append(file_info["path"])
        if missing_files:
            print("CẢNH BÁO: Phát hiện các file sau bị thiếu so với Manifest:", missing_files)
        else:
            print("Tất cả các tệp tin được liệt kê trong Manifest đều tồn tại đầy đủ!")
    except Exception as e:
        print("Lỗi khi đọc file Manifest:", e)
else:
    print("Không tìm thấy file BUNDLE_MANIFEST.json")


Đã tìm thấy BUNDLE_MANIFEST.json chứa 79 tệp cấu hình.
Lỗi khi đọc file Manifest: string indices must be integers, not 'str'


## 5. Nơi lưu và cấu hình
Drive được khuyên dùng để giữ checkpoint. Nếu không dùng Drive, cần tải kết quả về trước khi Colab kết thúc. Giữ nguyên cấu hình khi resume. Muốn đổi batch/epoch thì đổi RUN_ID để tạo một lần thử mới. Nếu hết VRAM, dùng BATCH_SIZE=4 và GRAD_ACC=4 trong một run mới.

In [18]:
USE_DRIVE = True
RUN_ID = "videberta_v1_01"
EPOCHS = 5
BATCH_SIZE = 8
GRAD_ACC = 2
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_DIR = Path("/content/drive/MyDrive/nlp_chuan_hoa_dia_chi") / RUN_ID
else:
    RUN_DIR = Path("/content/runs") / RUN_ID
print("Run:", RUN_DIR, "Resume:", RUN_DIR.exists())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Run: /content/drive/MyDrive/nlp_chuan_hoa_dia_chi/videberta_v1_01 Resume: False


## 6. Fine-tune GPU
Checkpoint pretrained được tải trực tiếp từ commit đã chốt. Lưu model và resume sau mỗi epoch. Resume quay lại epoch hoàn chỉnh gần nhất; nếu chưa xong epoch nào thì bắt đầu lại epoch 1. Không dùng dev để chọn checkpoint.

In [21]:
cmd = [sys.executable, "-u", "-m", "colab.train_videberta", "--data-root", "bundle_data", "--run-dir", str(RUN_DIR), "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE), "--gradient-accumulation", str(GRAD_ACC)]
if RUN_DIR.exists():
    cmd.append("--resume")
subprocess.run(cmd, check=True)
print((RUN_DIR / "training.json").read_text())


CalledProcessError: Command '['/usr/bin/python3', '-u', '-m', 'colab.train_videberta', '--data-root', 'bundle_data', '--run-dir', '/content/drive/MyDrive/nlp_chuan_hoa_dia_chi/videberta_v1_01', '--epochs', '5', '--batch-size', '8', '--gradient-accumulation', '2']' returned non-zero exit status 1.

## 7. Train ranker và đánh giá CPU
Ranker học trên phần train riêng; đo dev với luật v3, CRF cũ, ViDeBERTa dùng điểm tay và dùng ranker. **Độ trễ ở đây là CPU Colab**, không phải CPU laptop. Tỷ lệ khớp là với nhãn máy/tổng hợp, chưa phải gold. Muốn đo lại, đổi tên thư mục báo cáo để giữ lịch sử.

In [ ]:
REPORT_DIR = RUN_DIR / "reports_cpu_colab_01"
subprocess.run([sys.executable, "-u", "-m", "colab.evaluate_videberta", "--run-dir", str(RUN_DIR), "--output", str(REPORT_DIR)], check=True)
print((REPORT_DIR / "BAO_CAO.md").read_text())


## 8. Thử một địa chỉ trên CPU

In [ ]:
subprocess.run([sys.executable, "-m", "src", "Số 12, Đường Lê Lợi, Phường Trúc Bạch, Quận Ba Đình, Hà Nội", "--videberta-model", str(RUN_DIR / "model"), "--ranker", str(RUN_DIR / "ranker.txt"), "--threads", "1"], check=True)


## 9. Xuất kết quả thẳng về Drive
ZIP kết quả được lưu trong `My Drive/nlp_chuan_hoa_dia_chi/ket_qua/`. Drive for desktop sẽ đồng bộ về máy; không cần widget download. Optimizer/resume lớn vẫn nằm trong thư mục run để tiếp tục khi bị ngắt.

In [ ]:
from datetime import datetime
EXPORT_DIR = DRIVE_ROOT / "ket_qua"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_ZIP = EXPORT_DIR / (RUN_ID + "_ket_qua_" + datetime.now().strftime("%Y%m%d_%H%M%S") + ".zip")
subprocess.run([sys.executable, "-m", "colab.export_results", "--run-dir", str(RUN_DIR), "--output", str(RESULT_ZIP)], check=True)
print("Đã lưu kết quả trên Drive:", RESULT_ZIP)
